In [1]:
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials
from datetime import datetime

def connect_with_new_credentials():
    """Connect with fresh service account credentials"""
    try:
        # Use your NEW JSON file
        creds = Credentials.from_service_account_file(
            r"C:\Users\bunsong.fong\Code\Call_Query\khemra_account.json",  # ← Replace with your new file
            scopes=["https://www.googleapis.com/auth/spreadsheets"]
        )

        client = gspread.authorize(creds)
        print("✅ Connected with new credentials!")
        return client
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return None

# Use the new connection
gc = connect_with_new_credentials()

if gc:
    sheet = gc.open_by_key("1FeAYu8jgE_R7IWjcDPjhXsmXvpn79GbVAMa_WU0mxQs")

    # Check available worksheets
    print("Available worksheets:")
    for worksheet in sheet.worksheets():
        print(f" - {worksheet.title}")


✅ Connected with new credentials!
Available worksheets:
 - Sheet9
 - hq
 - retail_data
 - testing
 - Sheet22
 - call_data
 - call_users
 - Staff-Branch
 - List_Called
 - customerdata
 - cust_data
 - pw
 - FollowUp
 - branch_info
 - branch_data
 - followup_clean
 - Sheet21
 - ta_application
 - sale_plan
 - Users
 - cus_location
 - plan
 - actual
 - digital_data
 - closedeal
 - appointment
 - bis_partner
 - merchant_data
 - Sheet23
 - Copy of customerdata
 - call_log


In [2]:
# Access worksheet
ws = sheet.worksheet("testing")

# Load data
data = ws.get_all_records()

# Convert to DataFrame
df_retail = pd.DataFrame(data)

# Convert call_datetime column to datetime
df_retail["call_datetime"] = pd.to_datetime(df_retail["call_datetime"])

# Define date range
start_date = pd.to_datetime("2026-05-26")
end_date = pd.to_datetime("2026-05-26")

# Filter data (include the full last day)
df_retail_filtered = df_retail[
    (df_retail["call_datetime"] >= start_date) &
    (df_retail["call_datetime"] < end_date + pd.Timedelta(days=1))
]

print("✅ retail_data filtered successfully")
df_retail_filtered.head()


✅ retail_data filtered successfully


,call_id,call_datetime,customer_name,customer_phone,staff_id,caller_name,call_status,call_purpose,remark
2276,600545891,2026-05-26 08:06:00,SOPHAL,12221179,90024584,Nan Rina,Pick Up,Other,
2277,628847878,2026-05-26 08:06:28,SREY NIN,95323245,90024584,Nan Rina,Pick Up,Other,
2278,654847120,2026-05-26 08:06:54,MEYHEANG,86690690,90024584,Nan Rina,Pick Up,Other,
2279,720824404,2026-05-26 08:07:20,SOKUNTHEA,928000645,90024584,Nan Rina,Pick Up,Other,
2280,750435069,2026-05-26 08:07:50,CHANNY,10370246,90024584,Nan Rina,Pick Up,Loan,


In [3]:

# Load Staff-Branch mapping (keep Name for fallback lookup)
ws_branch = sheet.worksheet("Staff-Branch")
df_branch = pd.DataFrame(ws_branch.get_all_records())
df_branch = df_branch[["Personnel Number", "Name", "Branch"]].copy()
df_branch["Personnel Number"] = df_branch["Personnel Number"].astype(str).str.strip()
print(f"✅ Staff-Branch loaded: {len(df_branch)} records")


✅ Staff-Branch loaded: 134 records


In [4]:
import os 
os.makedirs(r"C:\Users\bunsong.fong\Code\Call_Query", exist_ok=True)
df_retail_filtered.to_excel(r"C:\Users\bunsong.fong\Code\Call_Query\retail_filtered_20.05.2026.xlsx", index=False)

In [5]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.gridspec import GridSpec
from datetime import datetime
import os

# ─── Join branch info ────────────────────────────────────────────────────────
df_work = df_retail_filtered.copy()
df_work["staff_id"] = df_work["staff_id"].astype(str).str.strip().replace({"nan": "", "None": ""})

# Backfill blank staff_ids using caller_name → staff_id from the full dataset
full_id_map = (
    df_retail
    .assign(staff_id=lambda d: d["staff_id"].astype(str).str.strip().replace({"nan": "", "None": ""}))
    .query("staff_id != ''")
    .drop_duplicates("caller_name")
    .set_index("caller_name")["staff_id"]
)
missing = df_work["staff_id"] == ""
df_work.loc[missing, "staff_id"] = df_work.loc[missing, "caller_name"].map(full_id_map)
df_work["staff_id"] = df_work["staff_id"].fillna("").astype(str).str.strip()

df_work = df_work.merge(df_branch, left_on="staff_id", right_on="Personnel Number", how="left")
df_work["Branch"] = df_work["Branch"].fillna("Unknown")

# Replace numeric caller_name (e.g. "90023413") with actual Name from df_branch
numeric_name = df_work["caller_name"].astype(str).str.match(r'^\d+$', na=False)
has_name = df_work["Name"].notna()
df_work.loc[numeric_name & has_name, "caller_name"] = (
    df_work.loc[numeric_name & has_name, "Name"].apply(lambda n: str(n).title())
)

# ── Name-based fallback: match login-style staff_id (e.g. "sreymeas.hem")
# against the Name column in df_branch (e.g. "Sreymeas HEM") ─────────────────
def _norm(s):
    import re
    return re.sub(r'\s+', '', str(s).lower().replace(".", " ").strip())

name_idx = df_branch.copy()
name_idx["_key"] = name_idx["Name"].apply(_norm)
name_idx = name_idx.drop_duplicates("_key").set_index("_key")

for i in df_work.index[df_work["Branch"] == "Unknown"]:
    key = _norm(df_work.at[i, "staff_id"])
    if key in name_idx.index:
        df_work.at[i, "Branch"]      = name_idx.at[key, "Branch"]
        df_work.at[i, "staff_id"]    = str(name_idx.at[key, "Personnel Number"])
        df_work.at[i, "caller_name"] = name_idx.at[key, "Name"].title()

# ─── Build summary tables ────────────────────────────────────────────────────
staff_summary = (
    df_work
    .groupby(["staff_id", "caller_name", "Branch"], sort=False)
    .size().reset_index(name="Calls")
    .sort_values("Calls", ascending=False)
    .reset_index(drop=True)
)
staff_summary.insert(0, "#", range(1, len(staff_summary) + 1))
staff_summary.columns = ["#", "Staff ID", "Staff Name", "Branch", "No. of Calls"]

branch_summary = (
    df_work
    .groupby("Branch", sort=False)
    .size().reset_index(name="Calls")
    .sort_values("Calls", ascending=False)
    .reset_index(drop=True)
)
branch_summary.insert(0, "#", range(1, len(branch_summary) + 1))
branch_summary.columns = ["#", "Branch", "No. of Calls"]

report_date_str  = df_work["call_datetime"].dt.date.iloc[0].strftime("%d %B %Y")
report_date_file = df_work["call_datetime"].dt.date.iloc[0].strftime("%d.%m.%Y")
total_calls      = len(df_work)
n_staff          = len(staff_summary)
n_branch         = len(branch_summary)

# ─── Colors ──────────────────────────────────────────────────────────────────
PAGE_BG      = "#F0F7F3"
HEADER_BG    = "#006B3C"
TABLE_HDR_BG = "#1A8A50"
TABLE_HDR_FG = "#FFFFFF"
ROW_EVEN     = "#E6F4EC"
ROW_ODD      = "#FFFFFF"
TOTAL_BG     = "#B8DEC8"
ACCENT       = "#004D2A"
BORDER_CLR   = "#A8D5B8"
MUTED_TEXT   = "#4A7A5E"
CHART_CLR    = "#2E9E60"
CHART_ACCENT = "#00A86B"

# ─── Page header renderer ────────────────────────────────────────────────────
def draw_header(ax, subtitle=None):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    ax.add_patch(patches.FancyBboxPatch(
        (0, 0), 1, 1, boxstyle="round,pad=0.04",
        facecolor=HEADER_BG, edgecolor="none", transform=ax.transAxes
    ))
    ax.add_patch(patches.Rectangle(
        (0, 0), 0.006, 1, facecolor=CHART_ACCENT, edgecolor="none", transform=ax.transAxes
    ))
    ax.text(0.03, 0.62, "Daily Call Report",
            transform=ax.transAxes, fontsize=20, fontweight="bold", color="white", va="center")
    ax.text(0.03, 0.22, f"Date:  {report_date_str}" + (f"  |  {subtitle}" if subtitle else ""),
            transform=ax.transAxes, fontsize=10, color="#A8D5B8", va="center")
    ax.text(0.97, 0.5, f"{total_calls:,}\nTotal Calls",
            transform=ax.transAxes, fontsize=16, fontweight="bold",
            color=CHART_ACCENT, ha="right", va="center", multialignment="center")

# ─── Bar chart renderer ──────────────────────────────────────────────────────
def render_bar_chart(ax, labels, values, title, fs_tick=9):
    labels = [str(l) for l in labels]
    ax.set_facecolor("#FAFFFE")
    bar_colors = [CHART_CLR if i % 2 == 0 else CHART_ACCENT for i in range(len(labels))]
    bars = ax.barh(labels, values, color=bar_colors, edgecolor="white", linewidth=0.5, height=0.72)
    max_val = max(values) if values else 1
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + max_val * 0.012,
                bar.get_y() + bar.get_height() / 2,
                str(val), va="center", ha="left", fontsize=fs_tick, color=ACCENT, fontweight="bold")
    ax.set_title(title, fontsize=13, fontweight="bold", color=HEADER_BG, pad=10)
    ax.set_xlabel("Number of Calls", fontsize=9, color=MUTED_TEXT)
    ax.tick_params(axis="y", labelsize=fs_tick, colors=ACCENT)
    ax.tick_params(axis="x", labelsize=8, colors=MUTED_TEXT)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color(BORDER_CLR)
    ax.spines["bottom"].set_color(BORDER_CLR)
    ax.set_xlim(0, max_val * 1.25)
    ax.invert_yaxis()
    ax.grid(axis="x", color=BORDER_CLR, linewidth=0.4, alpha=0.7)
    ax.set_axisbelow(True)

# ─── Table renderer ──────────────────────────────────────────────────────────
def render_table(ax, df, title, col_widths, fs_data=8.5, fs_title=12):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    n_rows, n_cols = len(df), len(df.columns)
    title_h  = 0.075
    header_h = 0.06
    total_h  = 0.055
    avail    = 1 - title_h - header_h - total_h - 0.025
    row_h    = avail / max(n_rows, 1)
    table_top = 1 - title_h - 0.012

    ax.add_patch(patches.FancyBboxPatch(
        (0.005, 1 - title_h + 0.004), 0.99, title_h - 0.007,
        boxstyle="round,pad=0.006", facecolor=TABLE_HDR_BG, edgecolor="none",
        transform=ax.transAxes, zorder=2
    ))
    ax.text(0.5, 1 - title_h / 2, title,
            transform=ax.transAxes, fontsize=fs_title, fontweight="bold",
            color="white", ha="center", va="center", zorder=3)

    xs = [0]
    for w in col_widths[:-1]:
        xs.append(xs[-1] + w)

    y = table_top
    for col_name, x, w in zip(df.columns, xs, col_widths):
        ax.add_patch(patches.Rectangle(
            (x, y - header_h), w, header_h,
            facecolor=HEADER_BG, edgecolor=BORDER_CLR, linewidth=0.5,
            transform=ax.transAxes, zorder=2
        ))
        ax.text(x + w / 2, y - header_h / 2, col_name,
                transform=ax.transAxes, fontsize=fs_data, fontweight="bold",
                color=TABLE_HDR_FG, ha="center", va="center", zorder=3)

    for row_i, (_, row) in enumerate(df.iterrows()):
        y_row = y - header_h - row_i * row_h
        bg = ROW_EVEN if row_i % 2 == 0 else ROW_ODD
        for col_i, (val, x, w) in enumerate(zip(row, xs, col_widths)):
            ax.add_patch(patches.Rectangle(
                (x, y_row - row_h), w, row_h,
                facecolor=bg, edgecolor=BORDER_CLR, linewidth=0.3,
                transform=ax.transAxes, zorder=2
            ))
            is_num = (col_i == 0 or col_i == n_cols - 1)
            ha    = "center" if is_num else "left"
            xtext = x + w / 2 if is_num else x + 0.012
            ax.text(xtext, y_row - row_h / 2, str(val),
                    transform=ax.transAxes, fontsize=fs_data, color=ACCENT,
                    ha=ha, va="center", zorder=3, clip_on=True)

    y_total = y - header_h - n_rows * row_h
    ax.add_patch(patches.Rectangle(
        (0, y_total - total_h), 1, total_h,
        facecolor=TOTAL_BG, edgecolor=BORDER_CLR, linewidth=0.5,
        transform=ax.transAxes, zorder=2
    ))
    ax.text(0.5, y_total - total_h / 2,
            f"TOTAL:  {df.iloc[:, -1].sum():,}  calls",
            transform=ax.transAxes, fontsize=fs_data + 1, fontweight="bold",
            color=HEADER_BG, ha="center", va="center", zorder=3)

    ax.add_patch(patches.FancyBboxPatch(
        (0, y_total - total_h), 1, table_top - (y_total - total_h),
        boxstyle="square,pad=0", facecolor="none",
        edgecolor=TABLE_HDR_BG, linewidth=1.3,
        transform=ax.transAxes, zorder=4
    ))

# ─── Output path ─────────────────────────────────────────────────────────────
os.makedirs(r"C:\Users\bunsong.fong\Code\Call_Query\output", exist_ok=True)
output_pdf = rf"C:\Users\bunsong.fong\Code\Call_Query\output\Call_Summary_{report_date_file}.pdf"

# ─── Dynamic figure height ────────────────────────────────────────────────────
hdr_h          = 1.2
branch_chart_h = max(2.5, n_branch * 0.45)
branch_table_h = max(2.2, n_branch * 0.42)
row1_h         = max(branch_chart_h, max(3.5, n_staff * 0.22))
page_h         = hdr_h + row1_h + branch_table_h + 1.5

with PdfPages(output_pdf) as pdf:

    fig = plt.figure(figsize=(15, page_h))
    fig.patch.set_facecolor(PAGE_BG)

    gs = GridSpec(
        3, 2, figure=fig,
        left=0.04, right=0.97, top=0.97, bottom=0.04,
        height_ratios=[hdr_h, row1_h, branch_table_h],
        width_ratios=[0.56, 0.44],
        hspace=0.28, wspace=0.18
    )

    # Header — full width
    draw_header(fig.add_subplot(gs[0, :]), subtitle="Call Summary")

    # Staff table — left column, spans both content rows to the bottom
    tbl_staff = fig.add_subplot(gs[1:, 0])
    tbl_staff.set_facecolor("white")
    render_table(tbl_staff, staff_summary,
                 "Call Summary by Staff",
                 col_widths=[0.04, 0.10, 0.40, 0.35, 0.11],
                 fs_data=max(7, min(9, int(180 / max(n_staff, 1)))))

    # Branch bar chart — top right
    branch_ax = fig.add_subplot(gs[1, 1])
    render_bar_chart(branch_ax,
                     branch_summary["Branch"].tolist(),
                     branch_summary["No. of Calls"].tolist(),
                     "Calls by Branch", fs_tick=9)

    # Branch table — bottom right
    tbl_branch = fig.add_subplot(gs[2, 1])
    tbl_branch.set_facecolor("white")
    render_table(tbl_branch, branch_summary,
                 "Call Summary by Branch",
                 col_widths=[0.06, 0.77, 0.17])

    fig.text(0.5, 0.012,
             f"Generated on {datetime.now().strftime('%d %B %Y,  %H:%M')}  —  Daily Call Report",
             ha="center", fontsize=7.5, color=MUTED_TEXT, style="italic")

    pdf.savefig(fig, facecolor=PAGE_BG, bbox_inches="tight")
    plt.close(fig)

print(f"✅ PDF saved — 1 page ({n_staff} staff, {n_branch} branches): {output_pdf}")


✅ PDF saved — 1 page (45 staff, 15 branches): C:\Users\bunsong.fong\Code\Call_Query\output\Call_Summary_26.05.2026.pdf
